# ERGT Colab Phase 3 Stable Base

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

This run treats raw ERGT as diagnostic and tests the stable candidate: `detached_d + sigmoid_cosine + clipped D + alpha warmup + real/random/shuffled controls`. Training scripts print live progress at every eval interval and save `progress_log.jsonl` for plots.

In [ ]:
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"
DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
DEVICE = "cuda"

BASELINE_CONFIG = "configs/baseline/phase3_stable_base_seed2027.json"
BASELINE_RESULT = "runs/phase0_baseline/phase3_stable_base_seed2027/baseline_results.json"
STABLE_OUTPUT_DIR = "runs/phase3_geo_attention/phase3_stable_base"
ALPHA_ZERO_CONFIG = "configs/ergt_v1/phase3_stable_base/alpha_zero_cosine_seed2027.json"
ALPHA_ZERO_RESULT = f"{STABLE_OUTPUT_DIR}/alpha_zero_cosine/metrics.json"

RUN_BASELINE = True
RUN_STABLE = True
RUN_COMPARISON = True
FORCE_RERUN = False

STABLE_RUNS = {
    "alpha_zero_cosine": {
        "family": "alpha_zero",
        "alpha": 0.0,
        "config": ALPHA_ZERO_CONFIG,
        "result": ALPHA_ZERO_RESULT,
    },
    "real_d_alpha_0_05_warmup_cosine": {
        "family": "real_d",
        "alpha": 0.05,
        "config": (
            "configs/ergt_v1/phase3_stable_base/"
            "real_d_alpha_0_05_warmup_cosine_seed2027.json"
        ),
        "result": f"{STABLE_OUTPUT_DIR}/real_d_alpha_0_05_warmup_cosine/metrics.json",
    },
    "real_d_alpha_0_1_warmup_cosine": {
        "family": "real_d",
        "alpha": 0.1,
        "config": "configs/ergt_v1/phase3_stable_base/real_d_alpha_0_1_warmup_cosine_seed2027.json",
        "result": f"{STABLE_OUTPUT_DIR}/real_d_alpha_0_1_warmup_cosine/metrics.json",
    },
    "random_d_alpha_0_05_warmup_cosine": {
        "family": "random_d",
        "alpha": 0.05,
        "config": (
            "configs/ergt_v1/phase3_stable_base/"
            "random_d_alpha_0_05_warmup_cosine_seed2027.json"
        ),
        "result": f"{STABLE_OUTPUT_DIR}/random_d_alpha_0_05_warmup_cosine/metrics.json",
    },
    "random_d_alpha_0_1_warmup_cosine": {
        "family": "random_d",
        "alpha": 0.1,
        "config": (
            "configs/ergt_v1/phase3_stable_base/"
            "random_d_alpha_0_1_warmup_cosine_seed2027.json"
        ),
        "result": f"{STABLE_OUTPUT_DIR}/random_d_alpha_0_1_warmup_cosine/metrics.json",
    },
    "shuffled_d_alpha_0_1_warmup_cosine": {
        "family": "shuffled_d",
        "alpha": 0.1,
        "config": (
            "configs/ergt_v1/phase3_stable_base/"
            "shuffled_d_alpha_0_1_warmup_cosine_seed2027.json"
        ),
        "result": f"{STABLE_OUTPUT_DIR}/shuffled_d_alpha_0_1_warmup_cosine/metrics.json",
    },
}

## 1. Runtime probe

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Set Colab hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

## 2. Clone/update repo and install dependencies

In [ ]:
project_path = Path(PROJECT_DIR)
if project_path.exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
repo_head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True)
print("repo:", repo_head.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[data,viz]"], check=True)

## 3. Prepare WikiText-2

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Helpers for live summaries and plots

In [ ]:
import matplotlib.pyplot as plt


def run_command(command):
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


def load_jsonl(path):
    path = Path(PROJECT_DIR) / path
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def progress_path_from_result(result_path):
    return str(Path(result_path).parent / "progress_log.jsonl")


def print_progress_summary(label, result_path):
    rows = load_jsonl(progress_path_from_result(result_path))
    if not rows:
        print(label, "has no progress log yet")
        return
    last = rows[-1]
    keys = [
        "step",
        "validation_loss",
        "best_validation_loss",
        "perplexity",
        "alpha_effective",
        "geo_to_qk_ratio",
        "tokens_per_second",
        "gpu_memory_gb",
    ]
    print(label, {key: last.get(key) for key in keys if key in last})


def plot_progress(run_map):
    series = []
    for label, result_path in run_map.items():
        rows = load_jsonl(progress_path_from_result(result_path))
        rows = [row for row in rows if row.get("validation_loss") is not None]
        if rows:
            series.append((label, rows))
    if not series:
        print("No progress data to plot yet.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for label, rows in series:
        steps = [row["step"] for row in rows]
        axes[0].plot(steps, [row.get("validation_loss") for row in rows], label=label)
        axes[1].plot(steps, [row.get("train_loss") for row in rows], label=label)
        geo = [row.get("geo_to_qk_ratio") for row in rows]
        if any(value is not None for value in geo):
            axes[2].plot(steps, geo, label=label)
    axes[0].set_title("validation_loss")
    axes[1].set_title("train_loss")
    axes[2].set_title("geo_to_qk_ratio")
    for axis in axes:
        axis.set_xlabel("step")
        axis.grid(True, alpha=0.25)
    axes[0].legend(fontsize=8)
    plt.show()

## 5. Run matched baseline

In [ ]:
baseline_result_path = Path(PROJECT_DIR) / BASELINE_RESULT
if RUN_BASELINE and (FORCE_RERUN or not baseline_result_path.exists()):
    run_command(
        [
            sys.executable,
            "experiments/train_baseline.py",
            "--config",
            BASELINE_CONFIG,
            "--data-dir",
            DATA_DIR,
            "--device",
            DEVICE,
        ]
    )
else:
    print("Skipping baseline; result exists or RUN_BASELINE is False:", BASELINE_RESULT)

print_progress_summary("baseline", BASELINE_RESULT)

## 6. Run Stable Base conditions

In [ ]:
completed = {"baseline": BASELINE_RESULT}
if RUN_STABLE:
    for label, spec in STABLE_RUNS.items():
        result_path = Path(PROJECT_DIR) / spec["result"]
        if result_path.exists() and not FORCE_RERUN:
            print("Skipping existing Stable Base run:", label, spec["result"])
        else:
            print("\n=== Stable Base condition:", label, "===", flush=True)
            run_command(
                [
                    sys.executable,
                    "experiments/train_ergt_v1.py",
                    "--config",
                    spec["config"],
                    "--data-dir",
                    DATA_DIR,
                    "--device",
                    DEVICE,
                ]
            )
        completed[label] = spec["result"]
        print_progress_summary(label, spec["result"])
        plot_progress(completed)
else:
    print("RUN_STABLE is False; no Stable Base condition executed.")

for label, spec in STABLE_RUNS.items():
    if not (Path(PROJECT_DIR) / spec["result"]).exists():
        raise FileNotFoundError(f"Missing Stable Base result for {label}: {spec['result']}")

## 7. Compare Stable Base

In [ ]:
if RUN_COMPARISON:
    command = [
        sys.executable,
        "experiments/compare_phase3_stable_base.py",
        "--baseline",
        BASELINE_RESULT,
        "--alpha-zero",
        ALPHA_ZERO_RESULT,
        "--output-dir",
        STABLE_OUTPUT_DIR,
    ]
    for _label, spec in STABLE_RUNS.items():
        if spec["family"] == "alpha_zero":
            continue
        command.extend(["--run", f"{spec['family']}:{spec['alpha']}:{spec['result']}"])
    run_command(command)
else:
    print("RUN_COMPARISON is False; no Stable Base comparison generated.")

## 8. Print report and archive light outputs

In [ ]:
report_path = Path(PROJECT_DIR) / STABLE_OUTPUT_DIR / "phase3_stable_base_results.json"
with report_path.open("r", encoding="utf-8") as handle:
    report = json.load(handle)
print(json.dumps(report, indent=2, sort_keys=True)[:12000])

light_root = Path("/content/ergt_phase3_stable_base_light")
if light_root.exists():
    shutil.rmtree(light_root)

include_roots = [
    Path("runs/phase0_baseline/phase3_stable_base_seed2027"),
    Path(STABLE_OUTPUT_DIR),
]
for relative_root in include_roots:
    source_root = Path(PROJECT_DIR) / relative_root
    if not source_root.exists():
        continue
    target_root = light_root / relative_root
    target_root.mkdir(parents=True, exist_ok=True)
    for path in source_root.rglob("*"):
        if not path.is_file() or path.suffix not in {".json", ".jsonl"}:
            continue
        destination = target_root / path.relative_to(source_root)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)
        print("included:", destination.relative_to(light_root))

light_archive = shutil.make_archive("/content/ergt_phase3_stable_base_light", "zip", light_root)
print("Light archive ready:", light_archive)